In [1]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [2]:
Processed_Dataset = "20260925_pallas_raw_P02_P07_CLARA"

In [3]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [5]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [6]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 17229, range: 442)> Size: 61MB
array([[0.43386608, 0.43152717, 0.42917386, ...,        nan,        nan,
               nan],
       [0.4338672 , 0.43152833, 0.42917506, ...,        nan,        nan,
               nan],
       [0.43386832, 0.4315295 , 0.42917627, ...,        nan,        nan,
               nan],
       ...,
       [0.51684823, 0.51481737, 0.51275213, ...,        nan,        nan,
               nan],
       [0.51685357, 0.51482148, 0.51275502, ...,        nan,        nan,
               nan],
       [0.51685895, 0.51482563, 0.51275795, ...,        nan,        nan,
               nan]], shape=(17229, 442))
Coordinates:
  * Time     (Time) datetime64[ns] 138kB 2026-09-25T00:54:34.314000010 ... 20...
  * range    (range) float64 4kB 104.3 119.3 134.2 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 17229, range:

In [7]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")